# Homework 6 - Trent Douglas

## Problem 1
- Compute the **Julian Date** (JD)  
- Using the **IERS Bulletin A** information from above, compute **UT1 – UTC**  
- Compute the **TT/TDB** Julian Date  


In [4]:
from math import *
from standards import *

UTC_year = 2026
UTC_month = 2
UTC_day = 28
UTC_hour = 18
UTC_minute = 22
UTC_second = 45

print(f"{'Input Time:':<30} 2026-02-28T18:22:45.000Z UTC")

J_date_midnight = (
    floor((1461*(UTC_year+4800+(UTC_month-14)/12))/4)
    +floor((367*(UTC_month-2-12*((UTC_month-14)/12)))/12)
    -floor((3*((UTC_year+4900+(UTC_month-14)/12)/100))/4)
    +UTC_day-32075
)
d = UTC_hour/24+UTC_minute/1440+UTC_second/86400-0.5
J_date = J_date_midnight + d
print(f"{'Julian Date:':<30} {J_date:.8f}")
MJD = J_date - 2400000.5
print(f"{'Modified Julian Date:':<30} {MJD:.8f}")
T = 2000 + (MJD - 51544.03)/365.242199
T = round(T, 8)
print(f"{'T in Besselian Years:':<30} {T:.8f}")
UT2_UT1 = 0.022*sin(2*pi*T) - 0.012*cos(2*pi*T) - 0.006*sin(4*pi*T) + 0.007*cos(4*pi*T)
print(f"{'UT2-UT1:':<30} {UT2_UT1:.8f}")
TAI_UTC = 37 #slide 37
print(f"{'TAI-UTC:':<30} {TAI_UTC:.8f}")
TT_UTC = TAI_UTC + 32.184
print(f"{'TT-UTC:':<30} {TT_UTC:.8f}")
UT1_UTC = 0.0640+0.00003*(MJD-61091) - (UT2_UT1)
print(f"{'UT1-UTC:':<30} {UT1_UTC:.9f}")
Sec_From_J2000 = round((J_date - 2451545.0)*86400)
print(f"{'Seconds from J2000:':<30} {Sec_From_J2000:.8f}")
TAI_Sec = Sec_From_J2000 + TAI_UTC
print(f"{'TAI Seconds:':<30} {TAI_Sec:.8f}")
JD_TT = J_date + TT_UTC/86400
print(f"{'Julian Date TT:':<30} {JD_TT:.8f}")
JD_TDB = JD_TT # slide 40
print(f"{'Julian Date TT:':<30} {JD_TDB:.8f}")

Input Time:                    2026-02-28T18:22:45.000Z UTC
Julian Date:                   2461100.26579861
Modified Julian Date:          61099.76579861
T in Besselian Years:          2026.16273756
UT2-UT1:                       0.00398609
TAI-UTC:                       37.00000000
TT-UTC:                        69.18400000
UT1-UTC:                       0.060276879
Seconds from J2000:            825574965.00000000
TAI Seconds:                   825575002.00000000
Julian Date TT:                2461100.26659935
Julian Date TT:                2461100.26659935


- Using the **low-fidelity (low-precision) formula**, compute the **TOD Sun vector**  
  - Units = **meters**  


<img src="Low-Precision_Sun_Formula.png" alt="Low-Precision_Sun_Formula" width=600/>

In [5]:
n = JD_TT-2451545
L = (280.460 + 0.9856474*n)%360
g = (357.528 + 0.9856003*n)%360
Ecliptic_lon = L + 1.915*sin(radians(g))+0.020*sin(2*radians(g))
Ecliptic_lat = 0
Obliqity_of_eliptic = 23.439 - 0.0000004*n
R = 1.00014-0.01671*cos(radians(g))-0.00014*cos(2*radians(g))

x = R*cos(radians(Ecliptic_lon))
y = R*cos(radians(Obliqity_of_eliptic))*sin(radians(Ecliptic_lon))
z = R*sin(radians(Obliqity_of_eliptic))*sin(radians(Ecliptic_lon))

m_in_au = 149597870700
sun_coordinates = Vector3(x*m_in_au, y*m_in_au, z*m_in_au)
print("Analytic Sun:")
print(sun_coordinates)


Analytic Sun:
Vector3(x=139416118937.9925, y=-46115677223.11736, z=-19989660221.598114)


- Using the **low-fidelity (low-precision) formula**, compute the **TOD Moon vector**  
  - Units = **meters**


<img src="Low-Precision_Moon_Formula.png" alt="Low-Precision_Moon_Formula" width=600/>

In [6]:
T = (JD_TT-2451545)/36525

Ecliptic_lon = 218.32+481267.881*T\
+ 6.29*sin(radians(135.0 + 477198.87*T)) - 1.27*sin(radians(259.3 - 413335.36*T))\
+ 0.66*sin(radians(235.7 + 890534.22*T)) + 0.21*sin(radians(269.9 + 954397.74*T))\
- 0.19*sin(radians(357.5 + 35999.05*T)) - 0.11*sin(radians(186.5 + 966404.03*T))

Ecliptic_lat = 5.13*sin(radians(93.3 + 483202.02*T)) + 0.28*sin(radians(228.2 + 960400.89*T))\
- 0.28*sin(radians(318.3 + 6003.15*T)) - 0.17*sin(radians(217.6 - 407332.21*T))

Pi = 0.9508 + 0.0518*cos(radians(135.0 + 477198.87*T)) + 0.0095*cos(radians(259.3 - 413335.36*T))\
+ 0.0078*cos(radians(235.7 + 890534.22*T)) + 0.0028*cos(radians(269.9 + 954397.74*T))
r = 1/sin(radians(Pi))
l = cos(radians(Ecliptic_lat))*cos(radians(Ecliptic_lon))
m = 0.9175*cos(radians(Ecliptic_lat))*sin(radians(Ecliptic_lon)) - 0.3978*sin(radians(Ecliptic_lat))
n = 0.3978*cos(radians(Ecliptic_lat))*sin(radians(Ecliptic_lon)) + 0.9175*sin(radians(Ecliptic_lat))

x = r*l
y = r*m
z = r*n

radius_earth = 6378137 #m

moon_coordinates = Vector3(x*radius_earth, y*radius_earth, z*radius_earth)
print("Analytic Moon:")
print(moon_coordinates)


Analytic Moon:
Vector3(x=-219309451.47784483, y=270419853.2536121, z=137190415.30107856)
